# Text-to-SQL: QLoRA-дообучение Qwen2.5-1.5B на бесплатной Colab T4

Ноутбук проходит весь цикл проекта:

1. подготовка данных (sql-create-context);
2. оценка **базовой** модели на своём бенчмарке и held-out выборке;
3. QLoRA-обучение (ориентировочно 40–90 минут на T4 для 15 000 примеров);
4. оценка **дообученной** модели;
5. (опционально) оценка API-модели для сравнения по качеству и цене;
6. сводная таблица, график и разбор ошибок.

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Код и зависимости

Запусти ячейку ниже. Появится кнопка **«Выбрать файлы»** — выбери на компьютере архив `text2sql-lora.zip`.

Когда проект будет на GitHub, можно не загружать архив: впиши ссылку на репозиторий в `REPO`, и код скачается сам.

In [ ]:
import os, zipfile

REPO = ""   # потом, когда проект будет на GitHub: "https://github.com/<логин>/text2sql-lora.git"

%cd /content
if REPO:
    !git clone -q $REPO text2sql-lora
else:
    from google.colab import files
    uploaded = files.upload()                      # выбери text2sql-lora.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall('/content')

%cd /content/text2sql-lora
!pip install -q -r requirements.txt && pip install -q -e .

### (Опционально) Google Drive
Colab стирает файлы после отключения. Чтобы адаптер и результаты не потерялись, подключи Drive —
в конце ноутбука они туда скопируются.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/text2sql-lora'

## 2. Данные

In [ ]:
!python -m t2s.data --out data
!python -m t2s.benchmark_db --out benchmark/dbs

## 3. Базовая модель (до обучения)
Обе модели оцениваются в одинаковой 4-битной квантизации — иначе сравнение нечестное.

In [ ]:
!python -m t2s.evaluate --name base --backend hf --model Qwen/Qwen2.5-1.5B-Instruct --load-in-4bit

## 4. Обучение
Для быстрой проверки, что всё работает, добавь `--set training.max_steps=50`.

In [ ]:
!python -m t2s.train --config configs/qlora_qwen1.5b.yaml

## 5. Дообученная модель

In [ ]:
!python -m t2s.evaluate --name lora --backend hf --model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter outputs/qwen1.5b-sql-lora/adapter --load-in-4bit

## 6. (Опционально) API-модель для сравнения

По умолчанию — бесплатный тариф **Groq** и модель `openai/gpt-oss-120b` (120 млрд параметров — в 80 раз больше нашей).

1. Получи ключ на [console.groq.com/keys](https://console.groq.com/keys) (Create API Key).
2. В Colab слева нажми значок ключа 🔑 → **Add new secret**: имя `GROQ_API_KEY`, значение — ключ. Включи переключатель доступа для блокнота.
3. Запусти ячейку. Бесплатный тариф ограничивает частоту запросов, поэтому прогон 120 вопросов идёт ~15 минут.

Цены в команде — платного тарифа Groq за 1 млн токенов. Они нужны, чтобы посчитать, сколько стоили бы запросы в продакшене; проверь актуальные на [groq.com/pricing](https://groq.com/pricing).
Подойдёт и любой другой OpenAI-совместимый провайдер: поменяй адрес, модель и цены.

In [ ]:
!git pull -q   # подтянуть свежий код из репозитория
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['OPENAI_BASE_URL'] = 'https://api.groq.com/openai/v1'

!python -m t2s.evaluate --name gpt-oss-120b --backend openai --model openai/gpt-oss-120b \
    --suites custom --reasoning-effort low --max-new-tokens 2048 --rpm 10 \
    --price-in 0.15 --price-out 0.60

## 7. Итоги

In [ ]:
!python -m t2s.report --results results
!python -m t2s.compare results/base_predictions.jsonl results/lora_predictions.jsonl

from IPython.display import Markdown, Image, display
display(Markdown(open('results/RESULTS.md').read().replace('![accuracy](accuracy.png)', '')))
display(Image('results/accuracy.png'))

## 8. Сохранение
Ячейка скачает на компьютер `results.zip` (таблица, график, разбор ошибок).
Распакуй его в папку проекта, чтобы получилась папка `text2sql-lora/results/`.
Если подключала Google Drive, туда же скопируются результаты и обученный адаптер.

In [ ]:
!cd results && zip -qr ../results.zip . && cd ..
from google.colab import files
files.download('results.zip')

if 'SAVE_DIR' in globals():   # только если подключала Google Drive
    !mkdir -p "$SAVE_DIR" && cp -r results outputs/qwen1.5b-sql-lora/adapter outputs/qwen1.5b-sql-lora/train_meta.json "$SAVE_DIR"/

### (Опционально) Загрузить адаптер на Hugging Face Hub
Тогда демо `app.py` можно запустить в Hugging Face Space, указав `ADAPTER=<твой-логин>/qwen1.5b-text2sql-lora`.

In [ ]:
from huggingface_hub import create_repo, login, upload_folder
login()
HF_REPO = 'YOUR_HF_USERNAME/qwen1.5b-text2sql-lora'
create_repo(HF_REPO, exist_ok=True)
upload_folder(repo_id=HF_REPO, folder_path='outputs/qwen1.5b-sql-lora/adapter')